In [3]:
import json
from openai import OpenAI
from tqdm import tqdm

In [1]:
import sys
sys.path.append('/Users/mn/Desktop/BridgeAthletics/DataPreProcessing')
from Functions.DataSetManipulation import json_read


In [3]:
##################################################################

In [4]:
with open("/Users/mn/Desktop/BridgeAthletics/config.json", "r") as config_file:
    config = json.load(config_file)
    api_key = config["OPENAI_API_KEY"]

client = OpenAI(api_key=api_key)

In [5]:
def run_chatgpt(prompt, client, model="gpt-3.5-turbo"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    return response.choices[0].message.content

In [ ]:
############################ REMOVING USELESS BLOCK NAMES ####################################

In [59]:
json_file = "/Users/mn/Desktop/BridgeAthletics/Dataset1_reps/initialdataset.json"
with open(json_file, "r") as file:
    json_data = json.load(file)

In [60]:
print(len(json_data))

1948


In [10]:
for entry in json_data[:5]:
    block_name = entry["input"]
    prompt = f"Does the workout title '{block_name}' clearly describe the focus or purpose of the workout? Reply only with 'yes' or 'no'."
    print(prompt)
    print("\nOutput:")
    print(">>", run_chatgpt(prompt, client))
    print("\n-------------------------")

Does the workout title 'Warm Up Complex' clearly describe the focus or purpose of the workout? Reply only with 'yes' or 'no'.

Output:
>> Yes

-------------------------
Does the workout title 'A Series' clearly describe the focus or purpose of the workout? Reply only with 'yes' or 'no'.

Output:
>> No

-------------------------
Does the workout title 'B Series' clearly describe the focus or purpose of the workout? Reply only with 'yes' or 'no'.

Output:
>> No

-------------------------
Does the workout title 'KB Warm Up' clearly describe the focus or purpose of the workout? Reply only with 'yes' or 'no'.

Output:
>> Yes

-------------------------
Does the workout title 'Strength Block' clearly describe the focus or purpose of the workout? Reply only with 'yes' or 'no'.

Output:
>> Yes

-------------------------


In [30]:
for i, entry in tqdm(enumerate(json_data), total=len(json_data)):
    block_name = entry["input"]
    prompt = f"Does the workout title '{block_name}' clearly describe the focus or purpose of the workout? Reply only with 'yes' or 'no'."
    json_data[i]["useful"] = run_chatgpt(prompt, client)

100%|███████████████████████████████████████| 1948/1948 [13:30<00:00,  2.40it/s]


In [63]:
n=0
for i in range(0,len(json_data)):
    if (json_data[i]["useful"]!="Yes") and (json_data[i]["useful"]!="No"):
        print(json_data[i]["useful"])
    if json_data[i]["useful"]=="No":
        n=n+1

print("no:",n,"yes:",len(json_data))

no: 252 yes: 1948


In [36]:
new_json_file = "/Users/mn/Desktop/BridgeAthletics/Dataset1_reps/dataset-usefulYesNo.json"
with open(new_json_file, "w") as file:
    json.dump(json_data, file, indent=1)  

In [62]:
json_file = "/Users/mn/Desktop/BridgeAthletics/Dataset1_reps/dataset-usefulYesNo.json"
with open(json_file, "r") as file:
    json_data = json.load(file)
print(len(json_data))

1948


In [55]:
filtered_data = [item for item in json_data if item["useful"] == "Yes"]
print(len(filtered_data))

1696


In [56]:
new_json_file = json_file.replace("-usefulYesNo.json", "-Yes.json")
with open(new_json_file, "w") as file:
    json.dump(filtered_data, file, indent=1) 

In [ ]:
####################### EXTRACTING KEY INFO FROM BLOCK NAMES #################################

In [56]:
json_file = "/Users/mn/Desktop/BridgeAthletics/Dataset1_reps/dataset-Yes.json"
with open(json_file, "r") as file:
    json_data = json.load(file)
print(len(json_data))

1696


In [57]:
print(json_data[0])

{'input': 'Warm Up Complex', 'output': [{'exercise': 'Barbell Front Squat', 'sets': 2, 'reps': [10, 10]}, {'exercise': 'Barbell RDL', 'sets': 2, 'reps': [10, 10]}], 'useful': 'Yes'}


In [8]:
for entry in json_data[:5]:
    block_name = entry["input"]
    prompt = f"For the workout title: '{block_name}', extract only the key phrases that describe the focus or type of the workout, expanding any abbreviations and removing any numerical details and session identifiers like 'day' or 'week'. Reply only with the title."
    print(prompt)
    print("\nOutput:")
    print(">>", run_chatgpt(prompt, client))
    print("\n-------------------------")

For the workout title: 'Warm Up Complex', extract only the key phrases that describe the focus or type of the workout, expanding any abbreviations and removing any numerical details and session identifiers like 'day' or 'week'. Reply only with the title.

Output:
>> Warm Up Complex

-------------------------
For the workout title: 'KB Warm Up', extract only the key phrases that describe the focus or type of the workout, expanding any abbreviations and removing any numerical details and session identifiers like 'day' or 'week'. Reply only with the title.

Output:
>> KB Warm Up

-------------------------
For the workout title: 'Strength Block', extract only the key phrases that describe the focus or type of the workout, expanding any abbreviations and removing any numerical details and session identifiers like 'day' or 'week'. Reply only with the title.

Output:
>> Strength Block

-------------------------
For the workout title: 'Warm Up', extract only the key phrases that describe the f

In [58]:
for i, entry in tqdm(enumerate(json_data), total=len(json_data)):
    block_name = entry["input"]
    prompt = f"For the workout title: '{block_name}', extract only the key phrases that describe the focus or type of the workout, expanding any abbreviations and removing any numerical details and session identifiers like 'day' or 'week'. Reply only with the title."
    json_data[i]["new_input"] = run_chatgpt(prompt, client)

100%|███████████████████████████████████████| 1696/1696 [14:10<00:00,  1.99it/s]


In [ ]:
###################### DATA CLEANING: REMOVING REMAINING BAD ENTRIES ##########################

In [59]:
import re
def contains_keywords_or_number(s):
    keywords = ["title", "week", "day", "phase"]
    # Check for keywords
    if any(keyword in s.lower() for keyword in keywords):
        return True
    # Check for numbers
    if re.search(r'\d', s):
        return True
    return False

In [81]:
inc_arr=[]
for i in range(len(json_data)):
    if contains_keywords_or_number(json_data[i]["new_input"]):
        print (i,json_data[i]["new_input"])
        inc_arr.append(i)
print([(i,j) for i,j in enumerate((inc_arr))])

1023 Big 4 Transmutation Lower
1024 Big 4 Transmutation Squats Depth Jump
1026 Big 4 Transmutation Military and Bench
1027 Big 4 Transmutation Giant Set Upper
1028 Big 4 Transmutation Lower
1030 Big 4 Transmutation Squats and Depth Jump
1033 Big 4 Transmutation Military and Bench
1034 Big 4 Transmutation Giant Set Upper
1035 Big 4 Transmutation Lower
1036 Big 4 Transmutation Squats and Depth Jump
1039 Big 4 Transmutation Military and Bench
1040 Big 4 Transmutation Giant Set Upper
1041 Big 4 Transmutation Lower
1043 Big 4 Transmutation Squats Depth Jump
1045 Big 4 Hypertrophy Lower Max Effort
1049 Big 4 Hypertrophy Presses
1050 Big 4 Hypertrophy Upper Accessory
1051 Big 4 Hypertrophy Squats and Depth Jump
1052 Big 4 Hypertrophy Lower Accessory
1053 Big 4 Hypertrophy Squats and Depth Jump
1054 Big 4 Hypertrophy Lower Accessory
1055 Big 4 Hypertrophy Presses
1056 Big 4 Hypertrophy Upper Accessory
1060 Big 4 Hypertrophy Lower Max Effort
1061 Big 4 Hypertrophy Squats and Depth Jump
1063 Big

In [80]:
# for i in range (0,len(json_data)):
#     if "Phase" in json_data[i]["new_input"]:
#         del(json_data[i])

# for i in range (0,len(json_data)):
#     if "Day" in json_data[i]["new_input"]:
#         del(json_data[i])

In [97]:
print(len(json_data))
print(json_data[0])

1637
{'input': 'Warm Up Complex', 'output': [{'exercise': 'Barbell Front Squat', 'sets': 2, 'reps': [10, 10]}, {'exercise': 'Barbell RDL', 'sets': 2, 'reps': [10, 10]}]}


In [85]:
for item in json_data:
    item['input'] = item['new_input']
    del item['new_input']

In [88]:
for item in json_data:
    del item['useful']

In [99]:
print(json_data[0])

{'input': 'Warm Up Complex', 'output': [{'exercise': 'Barbell Front Squat', 'sets': 2, 'reps': [10, 10]}, {'exercise': 'Barbell RDL', 'sets': 2, 'reps': [10, 10]}]}


In [148]:
filtered_json_data = []

for item in json_data:
    # Find isolated 'A', 'B', or 'C' in the 'input' field
    matches = re.findall(r'\b[A-C]\b', item['input'])
    if matches:
        print(item['input'])
    if not matches:
        filtered_json_data.append(item)
        
f_json_data = filtered_json_data
del filtered_json_data

Barbell Complex: No rest or changing weights B/W Exercises
Warm-Up B
Warm-Up A
Sprint Mechanics - NOT A SUPERSET
Superset A - Volume Accum.
PARTNER WORK A - ASSISTANCE
B Series - Hinge/ V-Pull
Upper Body French Contrast Max Effort A
Lower Body French Contrast Max Effort A
Block B- Upper Push-Pull
Block A Lower
Circuit C


In [149]:
print(len(f_json_data))

1625


In [187]:
filtered_json_data = [item for item in f_json_data if "finisher" not in item['input'].lower()]
f_json_data = filtered_json_data
del filtered_json_data

In [211]:
print(len(f_json_data))

1608


In [193]:
for item in f_json_data:
    if "finisher" in item["input"].lower():
        print (item)

In [192]:
new_json_file = "/Users/mn/Desktop/BridgeAthletics/Dataset1_reps/datasetprefinal.json"
with open(new_json_file, "w") as file:
    json.dump(f_json_data, file, indent=1)

In [ ]:
############################### REMOVING DUPLICATES ########################################

In [157]:
import argparse
from sklearn import __version__ as sklearn_version
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
json_file = "/Users/mn/Desktop/BridgeAthletics/Dataset1_reps/finaldataset.json"
with open(json_file, "r") as file:
    f_json_data = json.load(file)
print(len(f_json_data))

In [167]:
def remove_caps_punct(input):
    input = re.sub(r'[^\w\s]', '', input)
    input = input.lower()
    return input

In [291]:
def two_strings_cosine_similarity(string1, string2):
    vectorizer = TfidfVectorizer(stop_words=None, analyzer='char', ngram_range=(1, 3))
    tfidf_matrix = vectorizer.fit_transform([remove_caps_punct(string1).lower(), remove_caps_punct(string2).lower()])
    cos_sim_matrix = cosine_similarity(tfidf_matrix)
    return cos_sim_matrix[0, 1]

In [ ]:
def find_group_duplicates(json_data, threshold=0.9, key="input"):
    if key!="output":
        items = [remove_caps_punct(item[key]) for item in json_data]
    else:
        items = [remove_caps_punct(dict_list_to_string(item[key])) for item in json_data]
        
    vectorizer = TfidfVectorizer(stop_words=None, analyzer='char', ngram_range=(1, 3))
    tfidf = vectorizer.fit_transform(items)

    cos_sim_matrix = cosine_similarity(tfidf)

    dup_groups = {}
    unique_groups = {}
    visited = set()

    for i in range(len(cos_sim_matrix)):
        if i in visited:
            continue
            
        group = [i]
        for j in range(i + 1, len(cos_sim_matrix)):
            if cos_sim_matrix[i, j] > threshold:
                group.append(j)
                visited.add(j)
        
        if len(group) > 1:
            dup_groups[i] = group
            visited.add(i)

        else:
            unique_groups[i] = group
            
            
    duplicate_groups = [[json_data[id] for id in group] for group in dup_groups.values()]
    unique_groups = [[json_data[id] for id in group] for group in unique_groups.values()]

    return duplicate_groups, unique_groups


: 

In [347]:
def find_out_duplicates(json_data, key="output"):
    unique_out = []
    unique_obj = []
    k = 0
    output_dict = {}

    for item in json_data:
        output = item[key]
        output_str=json.dumps(item[key])
        input_block = item["input"]

        if output in unique_out:
            k += 1
            similar = False
            for existing_input in output_dict[output_str]:
                if two_strings_cosine_similarity(input_block, existing_input) >= 0.9:
                    similar = True
                    break
            if not similar:
                unique_obj.append(item)
                output_dict[output_str].append(input_block)
        else:
            unique_out.append(output)
            unique_obj.append(item)
            output_dict[output_str] = [input_block]

    return unique_out, unique_obj, k


In [315]:
duplicate_groups, unique_groups = find_group_duplicates(f_json_data, threshold=1, key="output")
t=0
if duplicate_groups:
    print("Duplicate Groups Found:")
    for group_id, group in enumerate(duplicate_groups, 1):
        # print(f"Group {group_id}:")
        # print([group[i]['input'] for i in range (0,len(group))])
        t=t+len(group)
        g=group_id
print("number of samples in duplicates:",t,"number of groups",g)

t=0
if unique_groups:
    print("Unique Groups Found:")
    for group_id, group in enumerate(unique_groups, 1):
        # print(f"Group {group_id}:")
        # print(group[0]['input'])
        # print(len(group))
        t=t+len(group)
        g=group_id
print("number of samples in uniques:",t,"number of groups",g)
        


Duplicate Groups Found:
number of samples in duplicates: 309 number of groups 103
Unique Groups Found:
number of samples in uniques: 1299 number of groups 1299


In [350]:
unique_outs,remove_dupli,k=find_out_duplicates(f_json_data)
f_json_data = remove_dupli

In [362]:
new_json_file = "/Users/mn/Desktop/BridgeAthletics/Dataset1_reps/finaldataset.json"
with open(new_json_file, "w") as file:
    json.dump(f_json_data, file, indent=1) 

In [4]:
json_file = "/Users/mn/Desktop/BridgeAthletics/Dataset1_reps/finaldataset.json"
with open(json_file, "r") as file:
    final_json_data = json.load(file)
print(len(final_json_data))

1367


In [5]:
# OPTIONAL Removing blocks that are too long
count=0
for i in range(len(final_json_data) - 1, -1, -1):
    if len(final_json_data[i]["output"]) >= 6:
        del final_json_data[i]
        count+=1
print(count)

57


In [7]:
new_json_file = "/Users/mn/Desktop/BridgeAthletics/Dataset1_reps/finaldataset_shortblocks.json"
with open(new_json_file, "w") as file:
    json.dump(final_json_data, file, indent=1)